In [3]:
import requests
import folium
from folium import plugins
import random
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import optuna

# ============================================================
# 0) SETTINGS
# ============================================================
optuna.logging.set_verbosity(optuna.logging.WARNING)
random.seed(42)
np.random.seed(42)

OUT_DIR = "hasil_optimasi_komparasi"
os.makedirs(OUT_DIR, exist_ok=True)

# ==========================================
# 1) DATASET - 31 LOKASI (SURABAYA)
# ==========================================
locations = {
    "1. Monumen Kapal Selam": {"coord": (-7.2654, 112.7503), "elev": 4.0},
    "2. Tugu Pahlawan": {"coord": (-7.2453, 112.7379), "elev": 9.0},
    "3. Jembatan Merah": {"coord": (-7.2365, 112.7383), "elev": 6.0},
    "4. Hotel Majapahit": {"coord": (-7.2573, 112.7388), "elev": 9.0},
    "5. Balai Kota Surabaya": {"coord": (-7.2589, 112.7469), "elev": 6.0},
    "6. House of Sampoerna": {"coord": (-7.2306, 112.7343), "elev": 6.0},
    "7. Museum Surabaya (Siola)": {"coord": (-7.2514, 112.7363), "elev": 7.0},
    "8. Rumah WR Soepratman": {"coord": (-7.2504, 112.7538), "elev": 5.0},
    "9. Gedung De Javasche Bank": {"coord": (-7.2365, 112.7358), "elev": 11.0},
    "10. Gereja Katolik Kepanjen": {"coord": (-7.2435, 112.7335), "elev": 6.0},
    "11. Jembatan Petekan": {"coord": (-7.222192, 112.738044), "elev": 5.0},
    "12. Gedung Internatio": {"coord": (-7.236281, 112.736915), "elev": 11.0},
    "13. Gedung Negara Grahadi": {"coord": (-7.263524, 112.743179), "elev": 7.0},
    "14. Kantor Pos Kebon Rojo": {"coord": (-7.243218, 112.737700), "elev": 6.0},
    "15. Rumah HOS Tjokroaminoto": {"coord": (-7.252464, 112.737747), "elev": 7.0},
    "16. Makam Belanda Peneleh": {"coord": (-7.253019586881285, 112.74035309946457), "elev": 5.0},
    "17. Kampung Lawang Seketeng": {"coord": (-7.250495592744265, 112.74075173450092), "elev": 7.0},
    "18. Gerbang Depan ITS": {"coord": (-7.279395032652996, 112.79009468032517), "elev": 3.0},
    "19. Gedung Cerutu": {"coord": (-7.2360970729302325, 112.73704713519352), "elev": 11.0},
    "20. Patung Karapan Sapi": {"coord": (-7.272209, 112.742095), "elev": 26.0},
    "21. Klenteng Sanggar Agung": {"coord": (-7.247199578801502, 112.80219558900477), "elev": 0.0},
    "22. Museum Pendidikan": {"coord": (-7.255254406638845, 112.74278515368215), "elev": 6.0},
    "23. Klenteng Hong Tiek Hian": {"coord": (-7.2367971558297395, 112.74387553040685), "elev": 5.0},
    "24. Monumen Bambu Runcing": {"coord": (-7.267096754354455, 112.74418425368475), "elev": 7.0},
    "25. Taman Prestasi": {"coord": (-7.261174085645066, 112.74291416110384), "elev": 5.0},
    "26. Penjara Kalisosok": {"coord": (-7.2343, 112.7351), "elev": 5.0},
    "27. Masjid Nasional Al-Akbar": {"coord": (-7.3381, 112.7148), "elev": 12.0},
    "28. Jembatan Suroboyo (Kenjeran)": {"coord": (-7.2515, 112.7964), "elev": 2.0},
    "29. Monumen Jenderal Sudirman": {"coord": (-7.2736, 112.7441), "elev": 8.0},
    "30. Kawasan Kota Tua Kembang Jepun": {"coord": (-7.2384, 112.7412), "elev": 6.0},
    "31. Pura Agung Jagat Karana": {"coord": (-7.2325, 112.7291), "elev": 5.0}
}

names = list(locations.keys())
coords_values = [loc["coord"] for loc in locations.values()]
elevations_list = [loc["elev"] for loc in locations.values()]
n = len(locations)
NUM_CITIES = n

START_CITY = "18. Gerbang Depan ITS"
start_location_idx = names.index(START_CITY)

print("=" * 90)
print("KOMPARASI ACO vs GA (Optuna + Fatigue-aware + Dashboard gabungan)")
print("=" * 90)

# ==========================================
# 2) OSRM DISTANCE MATRIX (CACHE)
# ==========================================
cache_path = os.path.join(OUT_DIR, "distance_matrix_km.npy")

if os.path.exists(cache_path):
    print("\n💾 Load distance matrix dari cache...")
    distance_matrix = np.load(cache_path)
else:
    print("\n📡 Mengambil Distance Matrix untuk SEPEDA dari OSRM...")
    coords_list = [f"{lon},{lat}" for lat, lon in coords_values]
    coords_string = ";".join(coords_list)
    url = f"http://router.project-osrm.org/table/v1/bike/{coords_string}?annotations=distance"
    response = requests.get(url, timeout=60).json()
    if response.get("code") != "Ok":
        raise RuntimeError(f"OSRM Error: {response}")
    distance_matrix = np.array(response["distances"]) / 1000.0
    np.save(cache_path, distance_matrix)
    print("✅ Distance Matrix SEPEDA OK!")

# ==========================================
# 3) CONSTRAINT / PENALTY
# ==========================================
LAMBDA_FATIGUE = 2.5
MU_VIOLATIONS = 10.0
RECOVERY_FACTOR = 0.82
EXTREME_EFFORT_M = 0.5  # ini kamu pakai; note: 0.5m sangat ketat

# ==========================================
# 4) METRICS + HELPERS
# ==========================================
def get_metrics(route):
    """
    route wajib format: [start, ..., start]
    """
    dist = 0.0
    fatigue = 0.0
    max_fatigue = 0.0
    violations = 0
    f_history = [0.0]

    for i in range(len(route) - 1):
        c = route[i]
        nxt = route[i + 1]
        dist += distance_matrix[c][nxt]

        elev_diff = elevations_list[nxt] - elevations_list[c]
        effort = max(0.0, elev_diff)

        if effort > EXTREME_EFFORT_M:
            violations += 1

        fatigue = (fatigue + effort) * RECOVERY_FACTOR
        max_fatigue = max(max_fatigue, fatigue)
        f_history.append(fatigue)

    score = dist + (LAMBDA_FATIGUE * max_fatigue) + (MU_VIOLATIONS * violations)
    return score, dist, max_fatigue, violations, f_history

def total_elevation_gain(route):
    gain = 0.0
    for i in range(len(route) - 1):
        c = route[i]
        nxt = route[i + 1]
        gain += max(0.0, elevations_list[nxt] - elevations_list[c])
    return gain

def compliance_percent(violations, legs):
    if legs <= 0:
        return 0.0
    return 100.0 * (legs - violations) / legs

# ==========================================
# 5) ACO (Optuna)
# ==========================================
print("\n" + "=" * 50)
print("🐜 TUNING ACO dengan Optuna")
print("=" * 50)

def run_aco(alpha, beta, evaporation, fatigue_sensitivity, num_ants, iterations, seed=123):
    random.seed(seed)
    np.random.seed(seed)

    pheromone = np.ones((NUM_CITIES, NUM_CITIES))
    best_score = float("inf")
    best_route = []
    TOP_K_RATIO = 0.2

    best_score_history = []

    for it in range(iterations):
        ants_routes = []
        ants_scores = []

        for _ in range(num_ants):
            route = [start_location_idx]
            current_f = 0.0
            visited = {start_location_idx}

            while len(route) < NUM_CITIES:
                curr = route[-1]
                candidates = [i for i in range(NUM_CITIES) if i not in visited]

                probs = []
                for cand in candidates:
                    d = distance_matrix[curr][cand]
                    effort = max(0.0, elevations_list[cand] - elevations_list[curr])
                    f_penalty = np.exp(current_f * fatigue_sensitivity) if effort > 0 else 1.0
                    vis = (1.0 / (d + 0.1)) / f_penalty
                    p = (pheromone[curr][cand] ** alpha) * (vis ** beta)
                    probs.append(p)

                sum_p = float(sum(probs))
                if sum_p <= 0:
                    next_c = random.choice(candidates)
                else:
                    pick = random.uniform(0, sum_p)
                    current_sum = 0.0
                    next_c = candidates[-1]
                    for idx, p in enumerate(probs):
                        current_sum += p
                        if current_sum >= pick:
                            next_c = candidates[idx]
                            break

                visited.add(next_c)
                route.append(next_c)
                current_f = (current_f + max(0.0, elevations_list[next_c] - elevations_list[curr])) * RECOVERY_FACTOR

            route.append(start_location_idx)  # cycle close

            s, _, _, _, _ = get_metrics(route)
            ants_routes.append(route)
            ants_scores.append(s)

            if s < best_score:
                best_score = s
                best_route = route[:]

        # pheromone update
        sorted_indices = np.argsort(ants_scores)
        top_k = max(1, int(TOP_K_RATIO * num_ants))
        pheromone *= (1.0 - evaporation)

        for idx in sorted_indices[:top_k]:
            r = ants_routes[idx]
            for j in range(NUM_CITIES):
                pheromone[r[j]][r[j + 1]] += 100.0 / (ants_scores[idx] + 0.1)

        if best_route:
            for j in range(NUM_CITIES):
                pheromone[best_route[j]][best_route[j + 1]] += 200.0 / (best_score + 0.1)

        best_score_history.append(best_score)

    return best_score, best_route, best_score_history

def objective_aco(trial):
    alpha = trial.suggest_float("alpha", 0.1, 1.0)
    beta = trial.suggest_float("beta", 4.0, 10.0)
    evaporation = trial.suggest_float("evaporation", 0.05, 0.97)
    fatigue_sensitivity = trial.suggest_float("fatigue_sensitivity", 0.05, 0.7)

    score, _, _ = run_aco(alpha, beta, evaporation, fatigue_sensitivity, num_ants=35, iterations=60, seed=123)
    return score

print("⏳ Optuna ACO (25 trials)...")
study_aco = optuna.create_study(direction="minimize")
study_aco.optimize(objective_aco, n_trials=25)
best_aco_params = study_aco.best_params
print("✅ Best ACO params:", best_aco_params)

print("\n▶ Menjalankan ACO Final (10 run)...")
NUM_RUNS_ACO = 10
aco_best_overall_score = float("inf")
aco_best_overall_route = []
aco_all_results = []

for run in range(1, NUM_RUNS_ACO + 1):
    s, r, hist = run_aco(
        alpha=best_aco_params["alpha"],
        beta=best_aco_params["beta"],
        evaporation=best_aco_params["evaporation"],
        fatigue_sensitivity=best_aco_params["fatigue_sensitivity"],
        num_ants=100,
        iterations=150,
        seed=1000 + run
    )

    s_final, d_final, f_final, v_final, _ = get_metrics(r)
    aco_all_results.append({
        "Algoritma": "ACO",
        "Run": run,
        "Score": s_final,
        "Dist_km": d_final,
        "MaxFatigue": f_final,
        "Violations": v_final,
        "Route": r[:],
        "ScoreHistory": hist[:]  # for dashboard
    })

    if s_final < aco_best_overall_score:
        aco_best_overall_score = s_final
        aco_best_overall_route = r[:]

# ==========================================
# 6) GA (Optuna)
# ==========================================
print("\n" + "=" * 50)
print("🧬 TUNING GA dengan Optuna")
print("=" * 50)

class GeneticAlgorithmFatigueTSP:
    def __init__(self, pop_size=120, max_gen=350, patience=60, min_improvement=0.0002,
                 mutation_rate=0.2, k_selection=5, seed=None):
        self.pop_size = pop_size
        self.max_gen = max_gen
        self.patience = patience
        self.min_improvement = min_improvement
        self.mutation_rate = mutation_rate
        self.k_selection = k_selection
        self.best_route = None
        self.best_score = float("inf")
        self.score_history = []

        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

    def create_individual(self):
        other = [i for i in range(n) if i != start_location_idx]
        random.shuffle(other)
        return [start_location_idx] + other + [start_location_idx]

    def fitness(self, route):
        score, *_ = get_metrics(route)
        return 1.0 / (1.0 + score)

    def selection(self, pop, fitness_scores):
        idxs = random.sample(range(len(pop)), self.k_selection)
        best = max(idxs, key=lambda i: fitness_scores[i])
        return pop[best].copy()

    def crossover(self, p1, p2):
        size = len(p1) - 1
        start = random.randint(1, size - 2)
        end = random.randint(start, size - 1)

        child = [None] * size
        child[0] = start_location_idx
        child[start:end + 1] = p1[start:end + 1]

        ptr = 1
        for gene in p2[1:-1] + p2[1:-1]:
            if gene not in child:
                while ptr < size and child[ptr] is not None:
                    ptr += 1
                if ptr >= size:
                    break
                child[ptr] = gene

        for gene in range(n):
            if gene not in child:
                for i in range(1, size):
                    if child[i] is None:
                        child[i] = gene
                        break

        return child + [start_location_idx]

    def mutate(self, route):
        if random.random() < self.mutation_rate:
            i, j = random.sample(range(1, len(route) - 1), 2)
            route[i], route[j] = route[j], route[i]
        return route

    def evolve(self):
        pop = [self.create_individual() for _ in range(self.pop_size)]
        no_improvement_count = 0
        last_best = float("inf")

        for _gen in range(self.max_gen):
            fitness_scores = [self.fitness(ind) for ind in pop]
            best_idx = int(np.argmax(fitness_scores))
            best_route_gen = pop[best_idx].copy()
            best_score_gen, *_ = get_metrics(best_route_gen)

            self.score_history.append(best_score_gen)

            if best_score_gen < self.best_score:
                self.best_score = best_score_gen
                self.best_route = best_route_gen.copy()

            improvement = (last_best - best_score_gen) / (last_best + 1e-12)
            if improvement > self.min_improvement:
                no_improvement_count = 0
                last_best = best_score_gen
            else:
                no_improvement_count += 1

            if no_improvement_count >= self.patience:
                break

            elite_size = min(15, self.pop_size)
            elite_idx = np.argsort(fitness_scores)[-elite_size:]
            elite = [pop[i].copy() for i in elite_idx]

            offspring = []
            while len(offspring) < (self.pop_size - elite_size):
                p1 = self.selection(pop, fitness_scores)
                p2 = self.selection(pop, fitness_scores)
                child = self.crossover(p1, p2)
                child = self.mutate(child)
                offspring.append(child)

            pop = elite + offspring

        return self.best_route, self.score_history

def objective_ga(trial):
    pop_size = trial.suggest_int("pop_size", 150, 300, step=10)
    mutation_rate = trial.suggest_float("mutation_rate", 0.01, 0.6)
    k_selection = trial.suggest_int("k_selection", 10, 15)

    ga = GeneticAlgorithmFatigueTSP(
        pop_size=pop_size,
        max_gen=120,
        patience=30,
        mutation_rate=mutation_rate,
        k_selection=k_selection,
        seed=123
    )
    best_route, _hist = ga.evolve()
    score, *_ = get_metrics(best_route)
    return score

print("⏳ Optuna GA (50 trials)...")
study_ga = optuna.create_study(direction="minimize")
study_ga.optimize(objective_ga, n_trials=50)
best_ga_params = study_ga.best_params
print("✅ Best GA params:", best_ga_params)

print("\n▶ Menjalankan GA Final (10 run)...")
NUM_RUNS_GA = 10
ga_best_overall_score = float("inf")
ga_best_overall_route = []
ga_all_results = []

for run in range(1, NUM_RUNS_GA + 1):
    ga = GeneticAlgorithmFatigueTSP(
        pop_size=best_ga_params["pop_size"],
        max_gen=350,
        patience=60,
        mutation_rate=best_ga_params["mutation_rate"],
        k_selection=best_ga_params["k_selection"],
        seed=2000 + run
    )
    best_route, hist = ga.evolve()
    s, d, f, v, _ = get_metrics(best_route)

    ga_all_results.append({
        "Algoritma": "GA",
        "Run": run,
        "Score": s,
        "Dist_km": d,
        "MaxFatigue": f,
        "Violations": v,
        "Route": best_route[:],
        "ScoreHistory": hist[:]  # for dashboard
    })

    if s < ga_best_overall_score:
        ga_best_overall_score = s
        ga_best_overall_route = best_route[:]

# ==========================================
# 7) KOMPARASI + CSV
# ==========================================
df_results = pd.DataFrame([{k: v for k, v in r.items() if k not in ("Route", "ScoreHistory")} for r in (aco_all_results + ga_all_results)])
csv_path = os.path.join(OUT_DIR, "komparasi_aco_vs_ga_optuna.csv")
df_results.to_csv(csv_path, index=False)

print("\n" + "=" * 60)
print("🏆 KESIMPULAN HASIL OPTIMASI (SETELAH TUNING OPTUNA)")
print("=" * 60)

aco_s, aco_d, aco_f, aco_v, _ = get_metrics(aco_best_overall_route)
ga_s, ga_d, ga_f, ga_v, _ = get_metrics(ga_best_overall_route)

print("🐜 ACO BEST:")
print(f"   Params      : {best_aco_params}")
print(f"   Score       : {aco_s:.2f}")
print(f"   Jarak       : {aco_d:.2f} km")
print(f"   Max Fatigue : {aco_f:.2f}")
print(f"   Violations  : {aco_v}")

print("\n🧬 GA BEST:")
print(f"   Params      : {best_ga_params}")
print(f"   Score       : {ga_s:.2f}")
print(f"   Jarak       : {ga_d:.2f} km")
print(f"   Max Fatigue : {ga_f:.2f}")
print(f"   Violations  : {ga_v}")

print("\n📂 CSV komparasi:", csv_path)

# ==========================================
# 8) DASHBOARD GABUNGAN (GA vs ACO) - 1 Gambar
# ==========================================
def build_df_for_dashboard(all_results):
    df = pd.DataFrame([{
        "Run": r["Run"],
        "Score": r["Score"],
        "Dist_km": r["Dist_km"],
        "MaxFatigue": r["MaxFatigue"],
        "Violations": r["Violations"],
    } for r in all_results])

    legs = n  # 31 segmen
    df["ElevationGain_m"] = [total_elevation_gain(r["Route"]) for r in all_results]
    df["Compliance_%"] = [compliance_percent(r["Violations"], legs) for r in all_results]
    return df

df_aco = build_df_for_dashboard(aco_all_results)
df_ga = build_df_for_dashboard(ga_all_results)

def plot_compare_dashboard(df_ga, df_aco, all_results_ga, all_results_aco, out_path):
    sns.set_style("whitegrid")

    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 3, wspace=0.25, hspace=0.35)

    m = min(len(df_ga), len(df_aco))
    x = np.arange(m)
    w = 0.38

    # (1) Distance per run
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.bar(x - w/2, df_ga["Dist_km"].iloc[:m], width=w, label="GA", color="cornflowerblue", edgecolor="black", alpha=0.85)
    ax1.bar(x + w/2, df_aco["Dist_km"].iloc[:m], width=w, label="ACO", color="lightskyblue", edgecolor="black", alpha=0.85)
    ax1.set_title("Total Distance per Run")
    ax1.set_xlabel("Run")
    ax1.set_ylabel("Distance (km)")
    ax1.set_xticks(x)
    ax1.set_xticklabels([str(i + 1) for i in range(m)])
    ax1.legend()

    # (2) Elevation gain
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.bar(x - w/2, df_ga["ElevationGain_m"].iloc[:m], width=w, label="GA", color="salmon", edgecolor="black", alpha=0.85)
    ax2.bar(x + w/2, df_aco["ElevationGain_m"].iloc[:m], width=w, label="ACO", color="lightsalmon", edgecolor="black", alpha=0.85)
    ax2.set_title("Total Elevation Gain")
    ax2.set_xlabel("Run")
    ax2.set_ylabel("Gain (m)")
    ax2.set_xticks(x)
    ax2.set_xticklabels([str(i + 1) for i in range(m)])
    ax2.legend()

    # (3) Compliance
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.bar(x - w/2, df_ga["Compliance_%"].iloc[:m], width=w, label="GA", color="seagreen", edgecolor="black", alpha=0.85)
    ax3.bar(x + w/2, df_aco["Compliance_%"].iloc[:m], width=w, label="ACO", color="mediumseagreen", edgecolor="black", alpha=0.85)
    ax3.set_ylim(0, 100)
    ax3.set_title("Constraint Compliance %")
    ax3.set_xlabel("Run")
    ax3.set_ylabel("%")
    ax3.set_xticks(x)
    ax3.set_xticklabels([str(i + 1) for i in range(m)])
    ax3.legend()

    # (4) Convergence best run
    ax4 = fig.add_subplot(gs[1, 0])
    best_ga = min(all_results_ga, key=lambda r: r["Score"])
    best_aco = min(all_results_aco, key=lambda r: r["Score"])
    ax4.plot(best_ga["ScoreHistory"], linewidth=2.2, label="GA (best run)", color="#005aab")
    ax4.plot(best_aco["ScoreHistory"], linewidth=2.2, label="ACO (best run)", color="#EA4335")
    ax4.set_title("Learning Convergence (Best Run)")
    ax4.set_xlabel("Generation / Iteration")
    ax4.set_ylabel("Score (lower is better)")
    ax4.legend()

    # (5) Violations
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.bar(x - w/2, df_ga["Violations"].iloc[:m], width=w, label="GA", color="mediumpurple", edgecolor="black", alpha=0.85)
    ax5.bar(x + w/2, df_aco["Violations"].iloc[:m], width=w, label="ACO", color="plum", edgecolor="black", alpha=0.85)
    ax5.set_title("Rule Violations (Lower is Better)")
    ax5.set_xlabel("Run")
    ax5.set_ylabel("Violations")
    ax5.set_xticks(x)
    ax5.set_xticklabels([str(i + 1) for i in range(m)])
    ax5.legend()

    # (6) Distance stability distribution
    ax6 = fig.add_subplot(gs[1, 2])
    tmp = pd.DataFrame({
        "Alg": (["GA"] * len(df_ga)) + (["ACO"] * len(df_aco)),
        "Dist_km": df_ga["Dist_km"].tolist() + df_aco["Dist_km"].tolist()
    })
    sns.boxplot(data=tmp, x="Alg", y="Dist_km", ax=ax6, palette=["cornflowerblue", "lightskyblue"])
    sns.stripplot(data=tmp, x="Alg", y="Dist_km", ax=ax6, color="#333333", size=5, jitter=0.12, alpha=0.7)
    ax6.set_title("Distance Stability Distribution")
    ax6.set_xlabel("")
    ax6.set_ylabel("Distance (km)")

    # (7) Compliance vs distance
    ax7 = fig.add_subplot(gs[2, 0])
    ax7.scatter(df_ga["Compliance_%"], df_ga["Dist_km"], s=120, edgecolor="black", alpha=0.85, label="GA")
    ax7.scatter(df_aco["Compliance_%"], df_aco["Dist_km"], s=120, edgecolor="black", alpha=0.85, label="ACO")
    ax7.set_title("Compliance vs Distance")
    ax7.set_xlabel("Compliance (%)")
    ax7.set_ylabel("Distance (km)")
    ax7.legend()

    # (8) Compliance density
    ax8 = fig.add_subplot(gs[2, 1])
    tmp2 = pd.DataFrame({
        "Alg": (["GA"] * len(df_ga)) + (["ACO"] * len(df_aco)),
        "Compliance_%": df_ga["Compliance_%"].tolist() + df_aco["Compliance_%"].tolist()
    })
    sns.violinplot(data=tmp2, x="Alg", y="Compliance_%", ax=ax8, palette=["seagreen", "mediumseagreen"], inner="box")
    ax8.set_title("Compliance Density")
    ax8.set_xlabel("")
    ax8.set_ylabel("Compliance (%)")

    # (9) Score distribution
    ax9 = fig.add_subplot(gs[2, 2])
    sns.kdeplot(df_ga["Score"], ax=ax9, linewidth=2.2, label="GA", color="#005aab")
    sns.kdeplot(df_aco["Score"], ax=ax9, linewidth=2.2, label="ACO", color="#EA4335")
    ax9.set_title("Score Distribution (KDE)")
    ax9.set_xlabel("Score")
    ax9.set_ylabel("Density")
    ax9.legend()

    fig.suptitle("GA vs ACO - Fatigue-Aware Comparison Dashboard", fontsize=16, fontweight="bold", y=0.99)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close(fig)

dashboard_path = os.path.join(OUT_DIR, "dashboard_compare_ga_vs_aco.png")
plot_compare_dashboard(df_ga, df_aco, ga_all_results, aco_all_results, dashboard_path)
print("✅ Dashboard saved:", dashboard_path)

# ==========================================
# 9) PETA FOLIUM KOMPARASI (layer ACO vs GA)
# ==========================================
print("\n🗺️ Membuat peta perbandingan rute ACO dan GA...")

center_lat = float(np.mean([c[0] for c in coords_values]))
center_lon = float(np.mean([c[1] for c in coords_values]))

m_compare = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="CartoDB positron")
fg_aco = folium.FeatureGroup(name="Rute ACO")
fg_ga = folium.FeatureGroup(name="Rute GA")

def get_osrm_geometry(route):
    if not route:
        return None

    # route already has start at end; OSRM butuh list without duplicated extra close? (boleh, tapi rapihin)
    route_indices = route[:]  # already closed
    route_coords_ordered = [f"{coords_values[i][1]},{coords_values[i][0]}" for i in route_indices]

    url_route = (
        f"http://router.project-osrm.org/route/v1/bike/"
        f"{';'.join(route_coords_ordered)}"
        f"?overview=full&geometries=geojson"
    )

    try:
        res = requests.get(url_route, timeout=30).json()
        if res.get("code") == "Ok":
            return [(lat, lon) for lon, lat in res["routes"][0]["geometry"]["coordinates"]]
    except Exception:
        return None
    return None

geom_aco = get_osrm_geometry(aco_best_overall_route)
if geom_aco:
    plugins.AntPath(locations=geom_aco, color="#F59E0B", pulse_color="white", weight=6, opacity=0.9, delay=800).add_to(fg_aco)
else:
    folium.PolyLine([coords_values[i] for i in aco_best_overall_route], color="#F59E0B", weight=4).add_to(fg_aco)

geom_ga = get_osrm_geometry(ga_best_overall_route)
if geom_ga:
    plugins.AntPath(locations=geom_ga, color="#3B82F6", pulse_color="white", weight=6, opacity=0.85, delay=1000).add_to(fg_ga)
else:
    folium.PolyLine([coords_values[i] for i in ga_best_overall_route], color="#3B82F6", weight=4, dash_array="10, 10").add_to(fg_ga)

for idx, coord in enumerate(coords_values):
    is_start = idx == start_location_idx
    folium.CircleMarker(
        location=coord,
        radius=6 if is_start else 4,
        color="green" if is_start else "#334155",
        fill=True,
        fill_opacity=0.8,
        tooltip=names[idx]
    ).add_to(m_compare)

fg_aco.add_to(m_compare)
fg_ga.add_to(m_compare)

# metrics for dashboard map UI
aco_time_h = int(aco_d / 20)
aco_time_m = int(((aco_d / 20) * 60) % 60)
ga_time_h = int(ga_d / 20)
ga_time_m = int(((ga_d / 20) * 60) % 60)

var_aco = fg_aco.get_name()
var_ga = fg_ga.get_name()

info_html = f"""
<style>
.leaflet-control-container {{ display: none !important; }}
.custom-dashboard {{
    position: fixed; bottom: 30px; left: 30px; width: 400px;
    background: rgba(255,255,255,0.95); backdrop-filter: blur(10px);
    border-radius: 15px; z-index: 9999; box-shadow: 0 10px 30px rgba(0,0,0,0.2);
    font-family: 'Segoe UI', sans-serif; overflow: hidden; border: 1px solid white;
}}
.dashboard-header {{ padding: 15px; background: #1e293b; color: white; text-align: center; font-weight: bold; }}
.toggle-box {{ display: flex; gap: 10px; padding: 15px; background: #f8fafc; border-bottom: 1px solid #e2e8f0; }}
.btn-t {{ flex: 1; padding: 10px; border: none; border-radius: 8px; cursor: pointer; font-weight: bold;
    transition: 0.3s; background: #cbd5e1; color: #64748b; }}
.btn-t.active-aco {{ background: #f59e0b; color: white; box-shadow: 0 4px 10px rgba(245,158,11,0.4); }}
.btn-t.active-ga {{ background: #3b82f6; color: white; box-shadow: 0 4px 10px rgba(59,130,246,0.4); }}
.metric-table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
.metric-table td, .metric-table th {{ padding: 10px 15px; border-bottom: 1px solid #f1f5f9; }}
.c-aco {{ color: #d97706; font-weight: bold; }}
.c-ga {{ color: #2563eb; font-weight: bold; }}
</style>

<div class="custom-dashboard">
    <div class="dashboard-header">📊 Perbandingan Rute ACO vs GA</div>
    <div class="toggle-box">
        <button id="tAco" class="btn-t active-aco">🐜 Tampilkan ACO</button>
        <button id="tGa" class="btn-t">🧬 Tampilkan GA</button>
    </div>

    <div style="padding: 10px 5px;">
        <table class="metric-table">
            <tr style="background: #f1f5f9;">
                <th style="text-align:left;">Metrik</th>
                <th class="c-aco" style="text-align:right;">ACO</th>
                <th class="c-ga" style="text-align:right; display:none;">GA</th>
            </tr>
            <tr><td>Fitness Score</td>
                <td class="c-aco" style="text-align:right;">{aco_s:.2f}</td>
                <td class="c-ga" style="text-align:right; display:none;">{ga_s:.2f}</td>
            </tr>
            <tr><td>Jarak Terbaik</td>
                <td class="c-aco" style="text-align:right;">{aco_d:.2f} km</td>
                <td class="c-ga" style="text-align:right; display:none;">{ga_d:.2f} km</td>
            </tr>
            <tr><td>Waktu Tempuh</td>
                <td class="c-aco" style="text-align:right;">{aco_time_h}h {aco_time_m}m</td>
                <td class="c-ga" style="text-align:right; display:none;">{ga_time_h}h {ga_time_m}m</td>
            </tr>
            <tr><td>Max Fatigue</td>
                <td class="c-aco" style="text-align:right;">{aco_f:.2f}</td>
                <td class="c-ga" style="text-align:right; display:none;">{ga_f:.2f}</td>
            </tr>
            <tr><td>Pelanggaran</td>
                <td class="c-aco" style="text-align:right; color:#ef4444;">{aco_v}</td>
                <td class="c-ga" style="text-align:right; color:#ef4444; display:none;">{ga_v}</td>
            </tr>
        </table>
    </div>
</div>

<div id="warningBox" style="
    display: none; position: fixed; top: 20px; left: 50%; transform: translateX(-50%);
    background: rgba(239, 68, 68, 0.95); color: white; padding: 12px 20px; border-radius: 10px;
    z-index: 99999; font-family: 'Segoe UI', sans-serif; font-size: 13px; font-weight: 600;
    box-shadow: 0 6px 20px rgba(0,0,0,0.2); backdrop-filter: blur(8px);
">⚠️ Pilih minimal satu algoritma untuk ditampilkan.</div>

<script>
document.addEventListener("DOMContentLoaded", function() {{
    var map = null;
    for (var key in window) {{
        if (window[key] instanceof L.Map) {{ map = window[key]; break; }}
    }}
    var acoLayer = window['{var_aco}'];
    var gaLayer = window['{var_ga}'];

    var acoActive = true;
    var gaActive = false;

    if(map && gaLayer) {{ map.removeLayer(gaLayer); }}

    function syncUI() {{
        document.querySelectorAll('.c-aco').forEach(e => {{
            e.style.display = acoActive ? 'table-cell' : 'none';
        }});
        document.querySelectorAll('.c-ga').forEach(e => {{
            e.style.display = gaActive ? 'table-cell' : 'none';
        }});
        document.getElementById('tAco').className = acoActive ? 'btn-t active-aco' : 'btn-t';
        document.getElementById('tGa').className = gaActive ? 'btn-t active-ga' : 'btn-t';

        if(map && acoLayer) {{
            if(acoActive) map.addLayer(acoLayer);
            else map.removeLayer(acoLayer);
        }}
        if(map && gaLayer) {{
            if(gaActive) map.addLayer(gaLayer);
            else map.removeLayer(gaLayer);
        }}

        const warningBox = document.getElementById('warningBox');
        if(!acoActive && !gaActive) warningBox.style.display = 'block';
        else warningBox.style.display = 'none';
    }}

    document.getElementById('tAco').onclick = function() {{
        acoActive = !acoActive; syncUI();
    }};
    document.getElementById('tGa').onclick = function() {{
        gaActive = !gaActive; syncUI();
    }};
    syncUI();
}});
</script>
"""

m_compare.get_root().html.add_child(folium.Element(info_html))

out_compare_html = os.path.join(OUT_DIR, "komparasi_peta_interaktif_final.html")
m_compare.save(out_compare_html)

print("✅ Peta komparasi saved:", out_compare_html)
print("✅ Output dashboard image:", dashboard_path)

KOMPARASI ACO vs GA (Optuna + Fatigue-aware + Dashboard gabungan)

💾 Load distance matrix dari cache...

🐜 TUNING ACO dengan Optuna
⏳ Optuna ACO (25 trials)...
✅ Best ACO params: {'alpha': 0.3957506182895993, 'beta': 4.34158906223622, 'evaporation': 0.6520152521079742, 'fatigue_sensitivity': 0.4118115859428178}

▶ Menjalankan ACO Final (10 run)...

🧬 TUNING GA dengan Optuna
⏳ Optuna GA (50 trials)...


/var/folders/lk/wc9vhhns1fn7k7j4sx2dl0t00000gn/T/ipykernel_62335/191975499.py:365: RuntimeWarning: invalid value encountered in scalar divide
  improvement = (last_best - best_score_gen) / (last_best + 1e-12)


✅ Best GA params: {'pop_size': 260, 'mutation_rate': 0.28589326486133815, 'k_selection': 13}

▶ Menjalankan GA Final (10 run)...

🏆 KESIMPULAN HASIL OPTIMASI (SETELAH TUNING OPTUNA)
🐜 ACO BEST:
   Params      : {'alpha': 0.3957506182895993, 'beta': 4.34158906223622, 'evaporation': 0.6520152521079742, 'fatigue_sensitivity': 0.4118115859428178}
   Score       : 167.69
   Jarak       : 81.92 km
   Max Fatigue : 18.31
   Violations  : 4

🧬 GA BEST:
   Params      : {'pop_size': 260, 'mutation_rate': 0.28589326486133815, 'k_selection': 13}
   Score       : 168.53
   Jarak       : 92.37 km
   Max Fatigue : 18.47
   Violations  : 3

📂 CSV komparasi: hasil_optimasi_komparasi/komparasi_aco_vs_ga_optuna.csv


/var/folders/lk/wc9vhhns1fn7k7j4sx2dl0t00000gn/T/ipykernel_62335/191975499.py:569: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=tmp, x="Alg", y="Dist_km", ax=ax6, palette=["cornflowerblue", "lightskyblue"])
/var/folders/lk/wc9vhhns1fn7k7j4sx2dl0t00000gn/T/ipykernel_62335/191975499.py:590: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=tmp2, x="Alg", y="Compliance_%", ax=ax8, palette=["seagreen", "mediumseagreen"], inner="box")
/var/folders/lk/wc9vhhns1fn7k7j4sx2dl0t00000gn/T/ipykernel_62335/191975499.py:605: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


✅ Dashboard saved: hasil_optimasi_komparasi/dashboard_compare_ga_vs_aco.png

🗺️ Membuat peta perbandingan rute ACO dan GA...
✅ Peta komparasi saved: hasil_optimasi_komparasi/komparasi_peta_interaktif_final.html
✅ Output dashboard image: hasil_optimasi_komparasi/dashboard_compare_ga_vs_aco.png
